In [12]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
import pandas as pd
import time

def get_coin_data(driver, coin):
    url = f'https://www.bitget.com/price/{coin}'
    driver.get(url)
    time.sleep(2)
    try:
        labels = driver.find_elements(By.XPATH, '//span[@class="text-[14px] text-[var(--content-secondary)]"]')
        values = driver.find_elements(By.XPATH, '//span[@class="text-[14px] font-[600]"]')
        price=driver.find_element(By.XPATH,'//span[@class="font-bold text-[40px] ltIpad:text-[32px] leading-[48px] ltIpad:leading-[38px] text-primaryText"]').text
        date=driver.find_element(By.XPATH,'//div[@class="text-[14px] mt-[24px] text-thirdText font-medium"]').text
        keys = [el.text.replace(':', '').strip() for el in labels]
        vals = [el.text.replace('\n', '').strip() for el in values]
        limit = min(len(keys), len(vals))
        data = {keys[i]: vals[i] for i in range(limit)}
        data['coin'] = coin  
        data['price'] = price
        data['date'] = date
        return data
    except Exception as e:
        print(f"Lỗi khi lấy dữ liệu {coin}: {e}")
        return {'coin': coin, 'error': str(e)}
list_coin=['bitcoin','ethereum','binance','solana','ripple','cardano',
'avalanche','polkadot','chainlink','tron']
driver = webdriver.Chrome()
all_data = [get_coin_data(driver, coin) for coin in list_coin]
driver.quit()
df = pd.DataFrame(all_data)

In [13]:

from datetime import datetime
import re
df['collection_timestamp'] = datetime.now().strftime('%Y-%m-%d %H:%M:%S')


def extract_date(date_string):
    try:
        date_pattern = r'(\w+ \d+, \d{4})'
        match = re.search(date_pattern, date_string)
        if match:
            parsed_date = datetime.strptime(match.group(1), '%B %d, %Y')
            return parsed_date.strftime('%Y-%m-%d')
        return None
    except Exception:
        return None

df['parsed_date'] = df['date'].apply(extract_date)
df_improved = df.copy()
priority_columns = ['coin', 'price', 'date', 'parsed_date', 'collection_timestamp']
other_columns = [col for col in df_improved.columns if col not in priority_columns and col != 'error']
if 'error' in df_improved.columns:
    df_improved = df_improved[priority_columns + other_columns + ['error']]
else:
    df_improved = df_improved[priority_columns + other_columns]
df_improved.set_index('coin', inplace=True)


In [14]:
df_improved

,price,date,parsed_date,collection_timestamp,Market cap,Fully diluted market cap,Volume (24h),24h volume / market cap,24h high,24h low,...,Circulating supply,Total supply,Circulation rate,Max supply,Price in BTC,Price in ETH,Price at BTC market cap,Price at ETH market cap,Contracts,Links
coin,,,,,,,,,,,,,,,,,,,,,
bitcoin,"$105,554.14",Last updated as of 2025-05-30 15:22:25（UTC+0）,None,2025-05-30 22:23:12,"$2,097,582,829,288.54","$2,097,582,829,288.54","$59,013,644,018.65",2.81%,"$107,137.43","$104,649.5",...,"19,872,104 BTC","19,872,103BTC",100.00%,"21,000,000BTC",1 BTC,40.49 ETH,"$105,554.14","$15,838.48",--,
ethereum,"₫67,840,938.08",Last updated as of 2025-05-30 15:22:30（UTC+0）,None,2025-05-30 22:23:12,"₫8,190,058,652,073,013","₫8,190,058,652,073,013","₫587,057,574,754,432",7.16%,"₫69,532,699.76","₫66,741,435.07",...,"120,724,430 ETH","120,724,433.41ETH",99.00%,--ETH,0.02470 BTC,1 ETH,"₫452,119,851.02","₫67,840,938.08",0xeeee...eeeeeee(Arbitrum)More,
binance,"₫17,454,185.61",Last updated as of 2025-05-30 15:22:35（UTC+0）,None,2025-05-30 22:23:12,"₫2,459,080,670,216,445.5","₫2,459,080,670,216,445.5","₫51,233,453,606,760.74",2.08%,"₫17,745,563.71","₫17,342,983.42",...,"140,887,730 BNB","140,887,734.33BNB",99.00%,--BNB,0.006355 BTC,0.2573 ETH,"₫387,414,219.77","₫58,131,807.39",0xeeee...eeeeeee(BNB Smart Chain (BEP20))More,
solana,"₫4,205,300.89",Last updated as of 2025-05-30 15:20:30（UTC+0）,None,2025-05-30 22:23:12,"₫2,189,468,735,139,038.2","₫2,189,468,735,139,038.2","₫107,453,996,184,189.33",4.90%,"₫4,425,065.14","₫4,174,438.28",...,"520,644,960 SOL","601,974,294.62SOL",86.00%,--SOL,0.001531 BTC,0.06199 ETH,"₫104,835,186","₫15,730,601.84",So1111...1111111(Solana)More,
ripple,"₫57,025.81",Last updated as of 2025-05-30 15:22:42（UTC+0）,None,2025-05-30 22:23:12,"₫3,346,617,259,238,903.5","₫3,346,617,259,238,903.5","₫95,068,677,543,297.08",2.84%,"₫59,559.34","₫56,169.56",...,"58,686,005,000 XRP","99,986,107,098XRP",58.00%,--XRP,0.{4}2076 BTC,0.0008406 ETH,"₫930,066.86","₫139,557.26",0x1d2f...6c60dbe(BNB Smart Chain (BEP20))More,
cardano,"₫18,386.15",Last updated as of 2025-05-30 15:22:46（UTC+0）,None,2025-05-30 22:23:12,"₫649,725,118,976,307.2","₫649,725,118,976,307.2","₫25,160,105,417,851.32",3.87%,"₫19,233.48","₫18,130.27",...,"35,337,744,000 ADA","44,994,501,828.19ADA",78.00%,--ADA,0.{5}6694 BTC,0.0002710 ETH,"₫1,544,578.29","₫231,765.18",0x3ee2...d435d47(BNB Smart Chain (BEP20))More,
avalanche,"₫565,140.47",Last updated as of 2025-05-30 15:22:50（UTC+0）,None,2025-05-30 22:23:12,"₫238,034,925,029,218.1","₫238,034,925,029,218.1","₫15,317,889,792,118.52",6.43%,"₫598,166.47","₫554,653.17",...,"421,196,030 AVAX","456,198,738.81AVAX",92.00%,"715,748,719AVAX",0.0002058 BTC,0.008330 ETH,"₫129,587,906.06","₫19,444,766.89",FvwEAh...GCgxN5Z(Avalanche X-Chain)More,
polkadot,"₫109,252.93",Last updated as of 2025-05-30 15:22:54（UTC+0）,None,2025-05-30 22:23:12,"₫172,881,094,469,499.94","₫172,881,094,469,499.94","₫7,460,768,790,048.34",4.31%,"₫117,465.33","₫108,004.39",...,"1,582,393,200 DOT","1,582,393,243.07DOT",99.00%,--DOT,0.{4}3978 BTC,0.001610 ETH,"₫34,493,267.06","₫5,175,741.76",0x7083...c873402(BNB Smart Chain (BEP20))More,
chainlink,"₫373,587.26",Last updated as of 2025-05-30 15:22:58（UTC+0）,None,2025-05-30 22:23:12,"₫245,484,176,846,164.4","₫245,484,176,846,164.4","₫16,923,480,266,054.13",6.89%,"₫402,936.59","₫372,600.16",...,"657,099,970 LINK","1,000,000,000LINK",65.00%,--LINK,0.0001360 BTC,0.005507 ETH,"₫83,064,851.49","₫12,463,946.08",0xf97f...8539FB4(Arbitrum)More,


In [15]:

if 'Market Cap' in df.columns:
    
    def convert_market_cap(value):
        if pd.isna(value):
            return None
        value = value.replace('$', '').replace(',', '')
        
        multiplier = 1
        if value.endswith('B'):
            multiplier = 1_000_000_000
            value = value[:-1]
        elif value.endswith('M'):
            multiplier = 1_000_000
            value = value[:-1]
        elif value.endswith('K'):
            multiplier = 1_000
            value = value[:-1]
            
        try:
            return float(value) * multiplier
        except:
            return None
    
    df['market_cap_value'] = df['Market Cap'].apply(convert_market_cap)
    
    
    market_cap_df = df.sort_values('market_cap_value', ascending=False)
    try:
        import matplotlib.pyplot as plt
        plt.figure(figsize=(12, 6))
        plt.bar(market_cap_df['coin'], market_cap_df['market_cap_value'])
        plt.title('Market Cap của các đồng tiền điện tử')
        plt.xlabel('Đồng tiền')
        plt.ylabel('Market Cap (USD)')
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
    except Exception as e:
        print(f"Không thể hiển thị biểu đồ: {e}")
        print("Dữ liệu Market Cap theo thứ tự giảm dần:")
        print(market_cap_df[['coin', 'Market Cap', 'market_cap_value']])

In [16]:
# Lưu dữ liệu với timestamp
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
filename = f'crypto_data_{timestamp}.csv'

# Lưu dữ liệu
df_improved.to_csv(filename)
print(f"Đã lưu dữ liệu vào file: {filename}")


try:
    excel_filename = f'crypto_data_{timestamp}.xlsx'
    df_improved.to_excel(excel_filename)
    print(f"Đã lưu dữ liệu vào file Excel: {excel_filename}")
except Exception as e:
    print(f"Không thể lưu file Excel: {e}")

Đã lưu dữ liệu vào file: crypto_data_20250530_222312.csv
Đã lưu dữ liệu vào file Excel: crypto_data_20250530_222312.xlsx


In [17]:
# Hàm cập nhật dữ liệu theo thời gian thực
def update_crypto_data(interval_minutes=15, max_updates=5):
    """
    Cập nhật dữ liệu tiền điện tử theo định kỳ
    
    interval_minutes: Thời gian giữa các lần cập nhật (phút)
    max_updates: Số lần cập nhật tối đa
    """
    driver = webdriver.Chrome()
    all_results = []
    
    try:
        for i in range(max_updates):
            print(f"Đang cập nhật lần {i+1}/{max_updates}...")
            
            # Thu thập dữ liệu mới
            current_data = [get_coin_data(driver, coin) for coin in list_coin]
            
            # Thêm timestamp
            timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
            for data in current_data:
                data['collection_timestamp'] = timestamp
            
            # Thêm vào kết quả
            all_results.extend(current_data)
            
            # Lưu dữ liệu
            temp_df = pd.DataFrame(current_data)
            temp_df.to_csv(f'crypto_data_update_{i+1}.csv')
            
            if i < max_updates - 1:
                # Đợi đến lần cập nhật tiếp theo
                print(f"Đợi {interval_minutes} phút cho lần cập nhật tiếp theo...")
                time.sleep(interval_minutes * 60)
    
    except Exception as e:
        print(f"Lỗi khi cập nhật dữ liệu: {e}")
    
    finally:
        driver.quit()
        
    # Tạo DataFrame từ tất cả kết quả
    all_updates_df = pd.DataFrame(all_results)
    return all_updates_df

# Để chạy cập nhật tự động, bỏ comment dòng dưới đây:
# updated_data = update_crypto_data(interval_minutes=15, max_updates=3)